https://github.tools.sap/IBUDefenseAndSecurity/s4_equipment_api 

In [1]:
## New function (more compact)

import pandas as pd
import requests
from requests.auth import HTTPBasicAuth

def query_s4_tables(base_url, username, password):
    # Helper function for making requests
    def make_request(url):
        response = requests.get(url, auth=HTTPBasicAuth(username, password))
        if response.status_code != 200:
            raise Exception(f"Failed to fetch data from {url}: {response.status_code}")
        return response.json().get('d', {}).get('results', [])

    try:
        # Query Functional Locations
        funcloc_url = base_url + "sap/opu/odata/sap/API_FUNCTIONALLOCATION/FunctionalLocation/?$format=json&$top=10000&sap-client=600"
        entity_data_1 = make_request(funcloc_url)
        df_locs = pd.DataFrame(entity_data_1)

        loc_cols_of_interest = [
            "FunctionalLocation", "FunctionalLocationLabelName", "FunctionalLocationName", 
            "FuncLocationStructure", "FunctionalLocationCategory", "SuperiorFunctionalLocation", 
            "FuncnlLocPosInSuperiorTechObj", "ManufacturerSerialNumber", "MaintenancePlant", 
            "FunctionalLocationHasEquipment", "FuncnlLocHasSubOrdinateFuncLoc"
        ]
        df_locs = df_locs[loc_cols_of_interest]  # Locations Table

        # Query Functional Locations with Equipment
        locseq_url = base_url + "sap/opu/odata/sap/API_FUNCTIONALLOCATION/FunctionalLocation/?$format=json&$top=10000&sap-client=600&$filter=FunctionalLocationHasEquipment eq 'X'"
        entity_data_2 = make_request(locseq_url)
        df_locs_eq = pd.DataFrame(entity_data_2)
        df_locs_eq = df_locs_eq[loc_cols_of_interest]  # New DF

        # Query Equipment
        eq_url = base_url + "sap/opu/odata/sap/API_EQUIPMENT/Equipment/?$format=json&$top=10000&sap-client=600&$filter=FunctionalLocation ne ''"
        entity_data_3 = make_request(eq_url)
        df_eq = pd.DataFrame(entity_data_3)

        eq_cols_of_interest = ["Equipment", "EquipmentName", "FunctionalLocation", "Material"]
        df_eq = df_eq[eq_cols_of_interest]

        # Join and remove full duplicates
        df_joined_eq = pd.merge(df_locs_eq, df_eq, on="FunctionalLocation").drop_duplicates()

        # Re-organize the table to be centered around equipment
        df_joined_eq = df_joined_eq[["Equipment", "EquipmentName", "Material", "FunctionalLocation", "FunctionalLocationName"]]

        # Added maintenance notifications
        notifs_url = base_url + "sap/opu/odata/sap/API_MAINTNOTIFICATION/MaintenanceNotification/?$format=json&sap-client=600&$top=10000"
        notif_data = make_request(notifs_url)
        df_notifs = pd.DataFrame(notif_data)

        notif_cols = ["MaintenanceNotification", "NotificationText", "NotificationType", "NotifProcessingPhase", "NotifProcessingPhaseDesc", "ConcatenatedActiveSystStsName",
                      "MaintenanceObjectIsDown",  "MaintPriority", "MaintPriorityDesc", 
                      "TechnicalObjectLabel", "TechObjIsEquipOrFuncnlLocDesc", "TechnicalObjectDescription",
                      "SuperiorTechnicalObject", "SuperiorTechnicalObjectName", "FunctionalLocation", "FunctionalLocationLabelName",
                      "CreationDateTime", "MalfunctionStartDateTime", "MalfunctionEndDateTime"]
        df_notifs = df_notifs[notif_cols]

        return df_locs, df_joined_eq, df_notifs

    except Exception as e:
        print(f"Error occurred: {e}")
        return None, None, None

df_locs, df_joined_eq, df_notifs = query_s4_tables("https://coeportal515.saphosting.de/", "ASEPA", "Password1")

In [2]:
df_locs.head()

,FunctionalLocation,FunctionalLocationLabelName,FunctionalLocationName,FuncLocationStructure,FunctionalLocationCategory,SuperiorFunctionalLocation,FuncnlLocPosInSuperiorTechObj,ManufacturerSerialNumber,MaintenancePlant,FunctionalLocationHasEquipment,FuncnlLocHasSubOrdinateFuncLoc
0,4322001-01,4322001-01,Hull,CH,M,,,1234–00–568–8901,5000,,X
1,4322001-01-01,4322001-01-01,Engine Compartment,CH,M,4322001-01,,1234–00–568–8903,5000,,
2,4322001-01-02,4322001-01-02,Commanders Position,CH,M,4322001-01,,1234–00–568–8904,5000,,
3,4322001-01-03,4322001-01-03,Drivers Position,CH,M,4322001-01,,1234–00–568–8907,5000,,
4,4322001-02,4322001-02,PowerPack,CH,M,,,1234–00–568–8915,5000,,X


In [3]:
df_joined_eq.head()

,Equipment,EquipmentName,Material,FunctionalLocation,FunctionalLocationName
0,10053507,Challenger 2 Min Battle Tank 001,,CH22001,Challenger 2 Main Battle Tank CR2
1,20000047,Challenger 2 Min Battle Tank CR2,,CH22001,Challenger 2 Main Battle Tank CR2
2,20000000,Hull,,CH22001-01,Hull
3,20000007,Turret,,CH22001-01-01,Turret
4,20000008,Engine Compartment,,CH22001-01-02,Engine Compartment


In [4]:
df_notifs.head()

,MaintenanceNotification,NotificationText,NotificationType,NotifProcessingPhase,NotifProcessingPhaseDesc,ConcatenatedActiveSystStsName,MaintenanceObjectIsDown,MaintPriority,MaintPriorityDesc,TechnicalObjectLabel,TechObjIsEquipOrFuncnlLocDesc,TechnicalObjectDescription,SuperiorTechnicalObject,SuperiorTechnicalObjectName,FunctionalLocation,FunctionalLocationLabelName,CreationDateTime,MalfunctionStartDateTime,MalfunctionEndDateTime
0,10000000,Low pressure on pump,M1,3,In Process,NOPR ORAS OSTS,False,,,10000131,Equipment,Main Oil Pump,CH47301-03-02-02-04,Main Oil Pump,CH47301-03-02-02-04,CH47301-03-02-02-04,/Date(1640276763000+0000)/,/Date(1640273276000+0000)/,None
1,10000001,Condition Check of CH47 Parts,M1,4,Completed,NOCO ORAS,False,,,CH47301,Functional Location,Boeing CH-47 Chinook - 301,,,CH47301,CH47301,/Date(1641462246000+0000)/,/Date(1641458593000+0000)/,None
2,10000002,Wheel Dismantling Notif.,M1,1,Outstanding,OSNO,False,4,4-Low,10000068,Equipment,23-909-02Prairie Masker Noise Suppressio,DDG52-06-05,Prairie-Masker,CH47301-01-01,CH47301-01-01,/Date(1641474393000+0000)/,/Date(1648457486000+0000)/,None
3,10000003,Dismantling Notif.,M1,1,Outstanding,OSNO,False,2,2-High,10000069,Equipment,U-4500-15Special Warfare Rigid-hull Infl,DDG52-09-01,#1 RIB,CH47301-01-01-01,CH47301-01-01-01,/Date(1641478030000+0000)/,/Date(1648457604000+0000)/,None
4,10000004,Installation Notification,M1,1,Outstanding,OSNO,False,2,2-High,10000069,Equipment,U-4500-15Special Warfare Rigid-hull Infl,DDG52-09-01,#1 RIB,,,/Date(1641478071000+0000)/,/Date(1648457604000+0000)/,None


In [5]:
def get_equipment_from_fl(funcLoc):

    try:
        df_loc_main = df_locs[df_locs["FunctionalLocation"] == funcLoc] ## FunctionalLocation has to be atomised, otherwise this will not work.

        fl_has_subs = df_loc_main["FuncnlLocHasSubOrdinateFuncLoc"].iloc[0] == "X" # True/False
        fl_has_eq = df_loc_main["FunctionalLocationHasEquipment"].iloc[0] == "X" # True/False

        if ((fl_has_subs == False) & (fl_has_eq == False)): # No subordinates, no equpiment
            return {"error": f"The chosen FunctionalLocation ({funcLoc}) has no subordinate functional locations or equipment attached."} # Return empty dictionary
        
        elif ((fl_has_subs == False) & (fl_has_eq == True)): # No subordinates, but equipment
            df_eq_subset = df_joined_eq[df_joined_eq["FunctionalLocation"]==funcLoc] # Find attached equipment
            dict_eq = df_eq_subset.to_dict(orient='records')
            return dict_eq # Return equipment dictionary
        
        elif ((fl_has_subs == True) & (fl_has_eq == False)): # No equpiment, but subordinates
            df_loc_subs = df_locs[df_locs["SuperiorFunctionalLocation"] == funcLoc] ## Find subordinate funcLocs

            df_eq_concat = pd.DataFrame()
            subordinate_ids = []  # List to store subordinate FunctionalLocation IDs
            # Iterate over rows in the dataframe of subordinate functional locations
            for _, row in df_loc_subs.iterrows():  
                sub_func_loc = row['FunctionalLocation']  # Access the FunctionalLocation value
                subordinate_ids.append(sub_func_loc)  # Store the subordinate FunctionalLocation ID
                
                # Recursively call the function for the subordinate functional location
                df_eq_subset = get_equipment_from_fl(sub_func_loc)  # Get equipment from the subordinate
                
                # Check if the result is a dictionary with equipment or error messages
                if isinstance(df_eq_subset, dict):
                    if "error" in df_eq_subset:
                        # Skip this iteration if an error is returned from the recursive call
                        continue
                    else:
                        # Otherwise, convert it into a DataFrame and concatenate
                        df_eq_concat = pd.concat([df_eq_concat, pd.DataFrame(df_eq_subset)], ignore_index=True)
            
            # If no valid data was added, return an empty dictionary or handle accordingly
            if df_eq_concat.empty:
                return {"error": f"No equipment found for the subordinates: {subordinate_ids}"}
            
            dict_eq = df_eq_concat.to_dict(orient='records')  # Convert the concatenated DataFrame to a dictionary (list of records)

            return dict_eq
        
        elif ((fl_has_subs == True) & (fl_has_eq == True)):  # Both subordinates and equipment
            df_loc_subs = df_locs[df_locs["SuperiorFunctionalLocation"] == funcLoc]  # Find subordinate funcLocs
            df_eq_subset = df_joined_eq[df_joined_eq["FunctionalLocation"] == funcLoc]  # Find equipment attached to the functional location

            # Initialize a DataFrame to accumulate equipment, starting with the functional location's own equipment
            df_eq_concat = df_eq_subset.copy()  # Start with the equipment at the functional location

            # Iterate over rows in the dataframe of subordinate functional locations
            for _, row in df_loc_subs.iterrows():
                sub_func_loc = row['FunctionalLocation']  # Access the FunctionalLocation value
                
                # Recursively call the function for the subordinate functional location
                df_eq_subset_sub = get_equipment_from_fl(sub_func_loc)  # Get equipment from the subordinate
                
                # Ensure that the result from the recursive call is a list of dictionaries or DataFrame
                if isinstance(df_eq_subset_sub, dict):
                    if "error" in df_eq_subset_sub:
                        # If there's an error for this subordinate, skip it
                        print("Skipping subordinate functional location due to error:", sub_func_loc)
                        continue
                    else:
                        # Otherwise, convert the dictionary to a DataFrame and concatenate it
                        df_eq_concat = pd.concat([df_eq_concat, pd.DataFrame([df_eq_subset_sub])], ignore_index=True)
                elif isinstance(df_eq_subset_sub, list):
                    # If it's a list of dictionaries (which is expected for the recursive call)
                    df_eq_concat = pd.concat([df_eq_concat, pd.DataFrame(df_eq_subset_sub)], ignore_index=True)
            
            # Convert the final concatenated DataFrame to a dictionary (list of records)
            dict_eq = df_eq_concat.to_dict(orient='records')  # Convert the DataFrame to a list of dictionaries

            return dict_eq
        
        else: 
            return {"error": "Please select a different FunctionalLocation"}
    except Exception as e:
        # In case of any failure, return a dictionary with default values
        print(f"Error occurred: {e}")  # Optional logging

In [6]:
get_equipment_from_fl("CH22001")

[{'Equipment': '10053507',
  'EquipmentName': 'Challenger 2 Min Battle Tank 001',
  'Material': '',
  'FunctionalLocation': 'CH22001',
  'FunctionalLocationName': 'Challenger 2 Main Battle Tank CR2'},
 {'Equipment': '20000047',
  'EquipmentName': 'Challenger 2 Min Battle Tank CR2',
  'Material': '',
  'FunctionalLocation': 'CH22001',
  'FunctionalLocationName': 'Challenger 2 Main Battle Tank CR2'},
 {'Equipment': '20000000',
  'EquipmentName': 'Hull',
  'Material': '',
  'FunctionalLocation': 'CH22001-01',
  'FunctionalLocationName': 'Hull'},
 {'Equipment': '20000007',
  'EquipmentName': 'Turret',
  'Material': '',
  'FunctionalLocation': 'CH22001-01-01',
  'FunctionalLocationName': 'Turret'},
 {'Equipment': '20000008',
  'EquipmentName': 'Engine Compartment',
  'Material': '',
  'FunctionalLocation': 'CH22001-01-02',
  'FunctionalLocationName': 'Engine Compartment'},
 {'Equipment': '20000009',
  'EquipmentName': 'Commanders Position',
  'Material': '',
  'FunctionalLocation': 'CH22001

In [7]:
# https://coeportal515.saphosting.de/sap/opu/odata/sap/API_MAINTNOTIFICATION/?$format=json&sap-client=600
# https://coeportal515.saphosting.de/sap/opu/odata/sap/API_MAINTNOTIFICATION/MaintenanceNotification/?$format=json&sap-client=600&$top=10

def get_notifs(funcLoc):
    ## Notifs

    df_funcloc = pd.DataFrame(get_equipment_from_fl(funcLoc)) # result df of get_equipment_from_fl
    limit_eq = list(set(df_funcloc["Equipment"]))
    limit_fl = list(set(df_funcloc["FunctionalLocation"]))

    notifs_dict = df_notifs[(df_notifs["TechnicalObjectLabel"].isin(limit_eq)) | df_notifs["FunctionalLocation"].isin(limit_fl)].to_dict(orient='records')
    return notifs_dict

In [8]:
get_notifs("CH22001")

[{'MaintenanceNotification': '10000640',
  'NotificationText': 'test',
  'NotificationType': 'M1',
  'NotifProcessingPhase': '1',
  'NotifProcessingPhaseDesc': 'Outstanding',
  'ConcatenatedActiveSystStsName': 'OSNO',
  'MaintenanceObjectIsDown': False,
  'MaintPriority': '3',
  'MaintPriorityDesc': '3-Medium',
  'TechnicalObjectLabel': 'CH22001',
  'TechObjIsEquipOrFuncnlLocDesc': 'Functional Location',
  'TechnicalObjectDescription': 'Challenger 2 Main Battle Tank CR2',
  'SuperiorTechnicalObject': '',
  'SuperiorTechnicalObjectName': '',
  'FunctionalLocation': 'CH22001',
  'FunctionalLocationLabelName': 'CH22001',
  'CreationDateTime': '/Date(1689847946000+0000)/',
  'MalfunctionStartDateTime': '/Date(1689847819000+0000)/',
  'MalfunctionEndDateTime': None},
 {'MaintenanceNotification': '10000641',
  'NotificationText': 'test',
  'NotificationType': 'M1',
  'NotifProcessingPhase': '1',
  'NotifProcessingPhaseDesc': 'Outstanding',
  'ConcatenatedActiveSystStsName': 'OSNO',
  'Mainte

---

## getSummary()

... summarize the results of `get_notifs` somehow

https://scc-internal-users.eu12.sapdas-staging.cloud.sap/joule

In [9]:
notifs_chinook = get_notifs("CH22001")
df_chinook = pd.DataFrame(notifs_chinook)
df_chinook

,MaintenanceNotification,NotificationText,NotificationType,NotifProcessingPhase,NotifProcessingPhaseDesc,ConcatenatedActiveSystStsName,MaintenanceObjectIsDown,MaintPriority,MaintPriorityDesc,TechnicalObjectLabel,TechObjIsEquipOrFuncnlLocDesc,TechnicalObjectDescription,SuperiorTechnicalObject,SuperiorTechnicalObjectName,FunctionalLocation,FunctionalLocationLabelName,CreationDateTime,MalfunctionStartDateTime,MalfunctionEndDateTime
0,10000640,test,M1,1,Outstanding,OSNO,False,3,3-Medium,CH22001,Functional Location,Challenger 2 Main Battle Tank CR2,,,CH22001,CH22001,/Date(1689847946000+0000)/,/Date(1689847819000+0000)/,None
1,10000641,test,M1,1,Outstanding,OSNO,False,3,3-Medium,CH22001,Functional Location,Challenger 2 Main Battle Tank CR2,,,CH22001,CH22001,/Date(1689848133000+0000)/,/Date(1689848056000+0000)/,None
2,10000642,test,Y1,3,In Process,APOK NOPR ORAS,False,,,CH22001,Functional Location,Challenger 2 Main Battle Tank CR2,,,CH22001,CH22001,/Date(1689848270000+0000)/,/Date(1689848729000+0000)/,None
3,10000669,PM intervention test,M1,3,In Process,NOPR ORAS,False,3,3-Medium,CH22001,Functional Location,Challenger 2 Main Battle Tank CR2,,,CH22001,CH22001,/Date(1691418136000+0000)/,/Date(1691418011000+0000)/,None
4,10000674,Ddc test,M1,1,Outstanding,OSNO,False,3,3-Medium,CH22001-01,Functional Location,Hull,CH22001,Challenger 2 Main Battle Tank CR2,CH22001-01,CH22001-01,/Date(1691671161000+0000)/,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2703,10003535,Environment temperature is too high,Y1,1,Outstanding,OSNO,False,2,2-High,20000047,Equipment,Challenger 2 Min Battle Tank CR2,,,CH22001,CH22001,/Date(1713176254000+0000)/,/Date(1713183454000+0000)/,None
2704,10003536,Environment temperature is too high,Y1,1,Outstanding,OSNO,False,2,2-High,20000047,Equipment,Challenger 2 Min Battle Tank CR2,,,CH22001,CH22001,/Date(1713176254000+0000)/,/Date(1713183454000+0000)/,None
2705,10003635,test,M1,3,In Process,NOPR ORAS,False,,,CH22001-01-01,Functional Location,Turret,CH22001-01,Hull,CH22001-01-01,CH22001-01-01,/Date(1715684950000+0000)/,None,None
2706,10003699,,M1,3,In Process,NOPR ORAS,False,2,2-High,20000001,Equipment,PowerPack,CH22001-02,PowerPack,CH22001-02,CH22001-02,/Date(1718032057000+0000)/,/Date(1718032137000+0000)/,None


Notification types:

- M1 - Maintenance Request (malfunction/problem notification)
- M2 - Malfunction Report (request for tasks)
- M3 - Activity Report (documentation of activities)

In [10]:
df_notifs["NotificationType"].value_counts()

NotificationType
Y1    2782
M1    1982
M2      28
Y3       6
Y2       3
I1       1
Name: count, dtype: int64

In [11]:
df_notifs["NotificationText"].value_counts()

NotificationText
Environment temperature is too high    2631
Dismantling Notif.                      137
                                        126
Installation Notification               122
Breakdown for CH2                        48
                                       ... 
Main Rotor hub bolt cracked               1
demo planning                             1
ufgf                                      1
#2 tire Worn and Cut                      1
Repair / Replace Fuel Pump                1
Name: count, Length: 576, dtype: int64

In [12]:
# Gets the top three of the Notification Texts that get shown

df_notifs["NotificationText"].value_counts().sort_values(ascending= False).head(3)

NotificationText
Environment temperature is too high    2631
Dismantling Notif.                      137
                                        126
Name: count, dtype: int64

In [13]:
# Gets the top three of Notification Type

df_notifs["NotificationType"].value_counts().sort_values(ascending=False).head(3)

NotificationType
Y1    2782
M1    1982
M2      28
Name: count, dtype: int64

In [14]:
#  Gets the top three of Notif Processing Pahse Desc

df_notifs["NotifProcessingPhase"].value_counts().sort_values(ascending= False).head(3)

NotifProcessingPhase
4    2970
1    1094
3     737
Name: count, dtype: int64

In [15]:
df_notifs["NotifProcessingPhaseDesc"].value_counts()

NotifProcessingPhaseDesc
Completed        2970
Outstanding      1094
In Process        737
Deletion Flag       1
Name: count, dtype: int64

ConcatenatedActiveSystStsName: 

- OSNO = Outstanding Notification
- NOPR = Notification in Process
- NOCO = Notification Completed 

In [16]:
df_notifs["ConcatenatedActiveSystStsName"].value_counts()

ConcatenatedActiveSystStsName
APRQ NOCO         2587
OSNO               997
NOPR ORAS          607
NOCO               191
NOCO ORAS          188
APRQ OSNO           93
NOPR                81
APOK NOPR           27
APOK NOPR ORAS      25
NOPR ORAS OSTS       1
APOK NOCO            1
DLFL NOCO            1
APOK NOCO ORAS       1
APRF NOCO            1
ATCO NOCO            1
Name: count, dtype: int64

In [17]:
df_notifs["MaintenanceObjectIsDown"].value_counts()

MaintenanceObjectIsDown
False    4662
True      140
Name: count, dtype: int64

In [18]:
df_notifs["MaintPriorityDesc"].value_counts()

MaintPriorityDesc
2-High         2830
               1596
3-Medium        245
1-Very high      69
4-Low            62
Name: count, dtype: int64

In [19]:
print("In", round((sum(df_notifs["MalfunctionEndDateTime"].isna()) / df_notifs.shape[0] ) * 100, 2),
      "% of cases, the 'MalfunctionEndDateTime' column is 'None'.") # is not always none

In 98.25 % of cases, the 'MalfunctionEndDateTime' column is 'None'.


GET A SUMMARY OF THE VALUES

In [20]:
import pandas as pd
# Idea: of each row get a dict of top three or percentages of each column

#   Not very relevant outputs: 
#   'MaintPriorityDesc': '2-High', 
#   'TechnicalObjectLabel': '20000047',
#   'TechObjIsEquipOrFuncnlLocDesc': 'Equipment',
#   'TechnicalObjectDescription': 'Challenger 2 Min Battle Tank CR2',
#   'SuperiorTechnicalObject': '',
#   'SuperiorTechnicalObjectName': '',


# -------------------------NOTIFICATION ANALYSIS--------------------------------


# Top three of the most Notification types, iterate through the subset of the dataframe and calculate for each type how many Notif Processing phase 
top_Three_Notifs_Types = df_notifs["NotificationType"].value_counts().sort_values(ascending= False).head(3)

df_notifs["NotifProcessingPhase"].value_counts().sort_values(ascending=False).head(3)

# 3 -> In process, 4 --> completed, 1 --> oustanding 
data = pd.DataFrame(df_notifs[["NotificationType", "NotifProcessingPhase", "NotifProcessingPhaseDesc"]])

# Data frame with only the top three, drop the other Notif types --> then put together all the types one after the other --> % of the proce phase 

not_Common_Values = df_notifs["NotificationType"].value_counts().sort_values().head(3)

# Dropping not common values form the dataframe

print(data[ data.NotificationType.isin(not_Common_Values) == False ])

top_Three_Notifs_Types_DataFrame = data[ data.NotificationType.isin(not_Common_Values) == False]


# How many times was each notif types completed?
# Drop other Processing phases that are not completed (4) 


     NotificationType NotifProcessingPhase NotifProcessingPhaseDesc
0                  M1                    3               In Process
1                  M1                    4                Completed
2                  M1                    1              Outstanding
3                  M1                    1              Outstanding
4                  M1                    1              Outstanding
...               ...                  ...                      ...
4797               M1                    1              Outstanding
4798               M1                    3               In Process
4799               M1                    1              Outstanding
4800               M1                    4                Completed
4801               M1                    4                Completed

[4802 rows x 3 columns]


In [21]:
not_Common_Values

NotificationType
I1    1
Y2    3
Y3    6
Name: count, dtype: int64

In [22]:
# Prints a dataframe with the notif types that are completed to check the % of success of each
values = [1, 2, 3]
completed_Notif_Types = top_Three_Notifs_Types_DataFrame[top_Three_Notifs_Types_DataFrame.NotifProcessingPhaseDesc == "Completed"]
completed_Notif_Types

,NotificationType,NotifProcessingPhase,NotifProcessingPhaseDesc
1,M1,4,Completed
5,M1,4,Completed
8,M1,4,Completed
9,Y1,4,Completed
10,M1,4,Completed
...,...,...,...
4768,M1,4,Completed
4772,M1,4,Completed
4785,M2,4,Completed
4800,M1,4,Completed


In [23]:

# Sum of all completed notifs types all together

sum = int(completed_Notif_Types["NotificationType"].value_counts().sum())
sum

2970

In [24]:
# Get value counts as a series
counts = df_notifs["NotificationType"].value_counts()

# Convert to DataFrame and reset index
df_counts = counts.reset_index()
df_counts.columns = ["NotificationType", "Count"]

# Add percentage column
total = df_counts["Count"].sum()
df_counts["Percentage"] = round( (df_counts["Count"] / total) * 100 )


print(df_counts)

# Data and Percentages in a dictionary

percentage_dict = dict(zip(df_counts["NotificationType"], df_counts["Percentage"]))
percentage_dict


  NotificationType  Count  Percentage
0               Y1   2782        58.0
1               M1   1982        41.0
2               M2     28         1.0
3               Y3      6         0.0
4               Y2      3         0.0
5               I1      1         0.0


{'Y1': 58.0, 'M1': 41.0, 'M2': 1.0, 'Y3': 0.0, 'Y2': 0.0, 'I1': 0.0}

In [25]:
# Top three of the dictionary

from itertools import islice

# Success rate of Notif types and only printing the top three most successful ones
top_three = dict(islice(percentage_dict.items(), 3))
top_three

{'Y1': 58.0, 'M1': 41.0, 'M2': 1.0}

### Location Dictionary Summary
Then merge Notification Success and Location 


In [26]:
# Top three of functional location names
# Malfunction start and end time how much time does it take until the malfunction is done? Will it be resolved? 
df_notifs["FunctionalLocationLabelName"].value_counts().sort_values(ascending = False).head(3) 

FunctionalLocationLabelName
CH22001    2692
           1406
CH47401     164
Name: count, dtype: int64

In [27]:
df_notifs["CreationDateTime"].value_counts().sort_values(ascending = False).head(3)

CreationDateTime
/Date(1717487378000+0000)/    5
/Date(1642607106000+0000)/    4
/Date(1645644661000+0000)/    4
Name: count, dtype: int64

In [28]:
df_notifs["MalfunctionStartDateTime"].value_counts().sort_values(ascending = False).head(3)

MalfunctionStartDateTime
/Date(1672531200000+0000)/    315
/Date(1640995200000+0000)/    315
/Date(1648457800000+0000)/     87
Name: count, dtype: int64

In [29]:
df_notifs["MalfunctionEndDateTime"].value_counts()

MalfunctionEndDateTime
/Date(1697774400000+0000)/    4
/Date(1695182400000+0000)/    3
/Date(1673478000000+0000)/    3
/Date(1681963200000+0000)/    3
/Date(1684555200000+0000)/    3
                             ..
/Date(1681272000000+0000)/    1
/Date(1694104560000+0000)/    1
/Date(1710217258000+0000)/    1
/Date(1681250400000+0000)/    1
/Date(1750600800000+0000)/    1
Name: count, Length: 64, dtype: int64

In [30]:
df_location = df_notifs[["MalfunctionEndDateTime", "MalfunctionStartDateTime", "FunctionalLocationLabelName"]]
df_location

,MalfunctionEndDateTime,MalfunctionStartDateTime,FunctionalLocationLabelName
0,None,/Date(1640273276000+0000)/,CH47301-03-02-02-04
1,None,/Date(1641458593000+0000)/,CH47301
2,None,/Date(1648457486000+0000)/,CH47301-01-01
3,None,/Date(1648457604000+0000)/,CH47301-01-01-01
4,None,/Date(1648457604000+0000)/,
...,...,...,...
4797,None,None,
4798,None,None,
4799,None,None,
4800,None,/Date(1741262973000+0000)/,


In [31]:
# Columns FunctionalLocationLabelName and FunctionalLocation have the same values

df_notifs['FunctionalLocationLabelName'].equals(df_notifs['FunctionalLocation'])

True

In [32]:
# Idea: check if the Malfunction has and end or not 

# MalfunctionEndDateTime != None

df_location.loc[df_location['MalfunctionEndDateTime'].notnull()]

,MalfunctionEndDateTime,MalfunctionStartDateTime,FunctionalLocationLabelName
387,/Date(1648155790000+0000)/,/Date(1648154999000+0000)/,CH47467
595,/Date(1694104560000+0000)/,/Date(1694023140000+0000)/,
635,/Date(1668830400000+0000)/,/Date(1668607601000+0000)/,CH22001
636,/Date(1671422400000+0000)/,/Date(1671199601000+0000)/,CH22001
637,/Date(1674100800000+0000)/,/Date(1673878001000+0000)/,CH22001
...,...,...,...
3429,/Date(1662847200000+0000)/,/Date(1662475588000+0000)/,LX10001
3430,/Date(1675465200000+0000)/,/Date(1675352788000+0000)/,LX10001
3548,/Date(1719547200000+0000)/,/Date(1719218579000+0000)/,RL002-02-02-02
3679,/Date(1592690400000+0000)/,/Date(1592648600000+0000)/,


In [33]:
# Getting the top three of the labels that have had an end time 
label_name = df_location["FunctionalLocationLabelName"].value_counts().head(3)
label_name

FunctionalLocationLabelName
CH22001    2692
           1406
CH47401     164
Name: count, dtype: int64